### **Week 6 Wednesday: Structured Error Handling with Custom Exceptions**

**Objective:** Learn to create and use custom exceptions to build more robust, maintainable, and user-friendly applications.

**Key Concepts:** Custom exceptions, exception hierarchies, API error handling, best practices

### **1. Why Custom Exceptions?**

### The Problem with Generic Exceptions:

```python
import requests

try:
    response = requests.get("https://api.example.com/data")
    response.raise_for_status()
except Exception as e:
    print(f"Error: {e}")  # Too generic! What kind of error?

    # Is it network? Authentication? Rate limiting? Data validation?
```

### Benefits of Custom Exceptions:

- **Specific error types** for different failure scenarios

- **Rich error information** (status codes, error details, timestamps)

- **Cleaner code** with targeted exception handling

- **Better debugging** with meaningful error messages

- **Domain-specific errors** that match your application logic

### **2. Creating Basic Custom Exceptions**

### Simple Custom Exception:

In [ ]:
class APIError(Exception):
    """Base exception for API-related errors"""
    pass

# Usage
def fetch_data():
    raise APIError("Failed to connect to API")

try:
    fetch_data()
except APIError as e:
    print(f"Custom error caught: {e}")

### Custom Exception with Additional Data:

In [ ]:
class APIError(Exception):
    """API error with status code and message"""
    
    def __init__(self, message, status_code=None, url=None):
        self.message = message
        self.status_code = status_code
        self.url = url
        super().__init__(self.message)
    
    def __str__(self):
        if self.status_code:
            return f"{self.message} (Status: {self.status_code}, URL: {self.url})"
        return self.message

# Usage
try:
    raise APIError("Rate limit exceeded", status_code=429, url="https://api.example.com/data")
except APIError as e:
    print(f"Error: {e}")
    print(f"Status code: {e.status_code}")
    print(f"URL: {e.url}")

### **3. Building Exception Hierarchies**

### Creating a Family of Related Exceptions:

In [ ]:
class APIError(Exception):
    """Base exception for all API errors"""
    def __init__(self, message, status_code=None, url=None):
        self.message = message
        self.status_code = status_code
        self.url = url
        super().__init__(self.message)

class RateLimitError(APIError):
    """Raised when API rate limit is exceeded"""
    def __init__(self, message="Rate limit exceeded", retry_after=None, **kwargs):
        self.retry_after = retry_after
        super().__init__(message, **kwargs)

class AuthenticationError(APIError):
    """Raised for authentication failures (401, 403)"""
    pass

class ServerError(APIError):
    """Raised for server errors (5xx)"""
    pass

class ClientError(APIError):
    """Raised for client errors (4xx, excluding auth)"""
    pass

# Test the hierarchy
try:
    
    raise RateLimitError(
        message="Too many requests", 
        status_code=429, 
        url="https://api.example.com/data",
        retry_after=60
    )
except RateLimitError as e:
    print(f"Rate limit error: {e}")
    print(f"Retry after: {e.retry_after} seconds")
except APIError as e:
    print(f"Generic API error: {e}")
except Exception as e:
    print(f"Other error: {e}")

### **4. Integrating with API Clients**

### Converting HTTP Errors to Custom Exceptions:

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

class APIClient:
    """API client with custom exception handling"""
    
    def __init__(self, base_url):
        self.base_url = base_url
        self.session = self._create_session()
    
    def _create_session(self):
        """Create session with retry logic"""
        session = requests.Session()
        
        retry_strategy = Retry(
            total=3,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
        )
        
        adapter = HTTPAdapter(max_retries=retry_strategy)
        session.mount("https://", adapter)
        session.mount("http://", adapter)
        
        return session
    
    def _handle_response(self, response):
        """Convert HTTP response to appropriate custom exception"""
        if response.ok:
            return response.json()
        
        # Map status codes to custom exceptions
        error_mapping = {
            401: AuthenticationError,   
            403: AuthenticationError,
            429: RateLimitError,
        }
        
        exception_class = error_mapping.get(response.status_code)
        
        if not exception_class:
            if 400 <= response.status_code < 500:
                exception_class = ClientError
            elif 500 <= response.status_code < 600:
                exception_class = ServerError
            else:
                exception_class = APIError
        
        # Extract error details from response
        try:
            error_data = response.json()
            message = error_data.get('error', response.reason)
        except:
            message = response.reason
        
        raise exception_class(
            message=message,
            status_code=response.status_code,
            url=response.url
        )
    
    def get_data(self, endpoint):
        """Make API request with custom exception handling"""
        url = f"{self.base_url}/{endpoint}"
        
        try:
            response = self.session.get(url, timeout=10)
            return self._handle_response(response)
            
        except requests.exceptions.Timeout:
            raise APIError("Request timed out", url=url)
        except requests.exceptions.ConnectionError:
            raise APIError("Connection failed", url=url)
        except requests.exceptions.RequestException as e:
            raise APIError(f"Request failed: {e}", url=url)

# Test the client
client = APIClient("https://httpbin.org")

try:
    # This will work
    data = client.get_data("json")
    print("Success:", data)
    
    # This will raise a custom exception
    client.get_data("status/429")
    
except RateLimitError as e:
    print(f"Rate limit error: {e}")
except AuthenticationError as e:
    print(f"Auth error: {e}")
except APIError as e:
    print(f"API error: {e}")

### **5. Advanced: Context Managers and Custom Exceptions**

### Creating a Resilient API Context Manager:

In [ ]:
class APIContext:
    """Context manager for API operations with automatic cleanup"""
    
    def __init__(self, base_url):
        self.base_url = base_url
        self.client = APIClient(base_url)
    
    def __enter__(self):
        print(f"Starting API session with {self.base_url}")
        return self.client
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        print("Closing API session")
        self.client.session.close()
        
        # Don't suppress exceptions
        return False

# Usage with context manager
try:
    with APIContext("https://httpbin.org") as api:
        data = api.get_data("json")
        print("Data received:", data)
        
except APIError as e:
    print(f"API operation failed: {e}")

# Session is automatically closed, even if exception occurs

### **6. Best Practices for Custom Exceptions**

### Do's and Don'ts:

| Do | Don't |
|----|--------|
| Create meaningful exception names | Use generic Exception for everything |
| Inherit from appropriate base classes | Create overly deep inheritance trees |
| Include relevant context in exceptions | Include sensitive data in error messages |
| Document when each exception is raised | Create exceptions for every minor case |
| Use exception hierarchies logically | Catch exceptions you can't handle |

### Good Exception Design:

In [ ]:
class WeatherAPIError(APIError):
    """Base exception for weather API errors"""
    
class LocationNotFoundError(WeatherAPIError):
    """Raised when weather location is not found"""
    def __init__(self, location, **kwargs):
        message = f"Weather location not found: {location}"
        super().__init__(message, **kwargs)

class InvalidDateError(WeatherAPIError):
    """Raised when date is invalid for weather data"""
    
class WeatherServiceUnavailableError(WeatherAPIError):
    """Raised when weather service is down"""

# Clean, targeted exception handling
def get_weather(location, date):
    try:
        # API call logic here
        if not location:
            raise LocationNotFoundError(location)
        # ... more logic
        
    except LocationNotFoundError:
        # Show user-friendly message
        print("Please check the location name")
        raise  # Re-raise for caller to handle
    except WeatherServiceUnavailableError:
        # Log and retry later
        print("Weather service is temporarily unavailable")
        raise

### **7. Quiz**

1. What's the main advantage of using custom exceptions over built-in ones?

2. How do you create an exception hierarchy?

3. When should you create a new custom exception vs using an existing one?

4. What information should you include in custom exceptions for API errors?

5. Why is it better to catch specific exceptions rather than generic ones?

**Answers:**

1. Domain-specific meaning, rich error information, better debugging

2. Create base exception and inherit from it for specific cases

3. When you need to handle a specific error case differently

4. Status code, URL, error message, timestamp, retry information

5. Prevents catching unrelated errors, enables targeted handling

### **8. Class Exercise: Build a Resilient Weather API Client**

**Task:** Create a weather API client with comprehensive custom exception handling.

### Requirements:
1. Create a custom exception hierarchy for weather-related errors

2. Implement an API client that converts HTTP errors to custom exceptions

3. Add retry logic for transient errors

4. Create a context manager for the API client

### Starter Code:

In [ ]:
# Exercise: Complete the WeatherAPI client

# Step 1: Define your custom exception hierarchy
class WeatherError(Exception):
    pass

# Add more specific exceptions here...

# Step 2: Implement the WeatherAPI client
class WeatherAPI:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://api.openweathermap.org/data/2.5"
        # Add session setup with retries
    
    def get_current_weather(self, city):
        # Implement with proper error handling
        # Convert HTTP errors to your custom exceptions
        pass

# Step 3: Test your implementation
def test_weather_api():
    api = WeatherAPI("your_api_key")
    
    try:
        weather = api.get_current_weather("London")
        print("Weather data:", weather)
    except WeatherError as e:
        print(f"Weather error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

# test_weather_api()